# Experiment 2: real fine-tuning of TinyNet for finger-vein PAD

Experiment 1 only used **frozen** ImageNet features plus k-NN. This notebook **trains the network weights** (end-to-end fine-tuning) and compares three regimes:

| Mode | What is trained | Backbone BatchNorm |
|---|---|---|
| `frozen` | only the new classifier head (reference = Experiment 1 style) | eval (fixed) |
| `partial` | head + the last 2 backbone stages + final conv | trainable stages in train mode |
| `full` | **every layer** of the network | train mode (statistics are re-estimated) |

Model: **TinyNet** (`timm`), a scaled-down EfficientNet. `tinynet_e` has only ~0.76 M backbone parameters (the classifier head adds ~0.33 M), so it is fast enough to fine-tune in every fold.

To prove the backbone really changes, each run logs `trainable_params` and `backbone_rel_change` = ||θ_after − θ_before|| / ||θ_before||. This value is 0 for `frozen` and > 0 for `partial` and `full`.

**Outputs** (in `/kaggle/working`, also in the *Output* tab after *Save Version*):
- `tinynet_runs.csv`: one row per run (dataset × size × fold × seed × mode), with all metrics
- `tinynet_summary.csv`: mean ± std per dataset × mode × size
- `tinynet_predictions.csv`: per-image test scores, for ROC/DET plots

**Settings:** Accelerator **GPU T4 x2** or **P100**, Internet **On**, input dataset **fvpad-data**.

In [ ]:
# 0. Environment
!nvidia-smi --query-gpu=name,memory.total --format=csv
try:
    import timm
except ImportError:
    !pip install -q timm
    import timm
import glob, os, json, time, copy
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())

hits = glob.glob('/kaggle/input/**/metadata.csv', recursive=True) or glob.glob('fvpad_data/metadata.csv')
assert hits, 'metadata.csv not found - did you add the fvpad-data dataset with "Add Input"?'
DATA = os.path.dirname(hits[0])
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print('DATA =', DATA, '| OUT =', OUT)

## 1. Configuration
Change only this cell. The defaults run the whole grid (3 datasets × 3 sizes × 5 folds × 1 seed × 3 modes = 135 runs). On a T4 this takes about **1.5–3 h**, so use *Save Version → Save & Run All* and close the browser.
For a quick test first, set `QUICK = True` (1 fold, BA only, 3 epochs: about 3 min).

In [ ]:
MODEL      = 'tinynet_e'          # tinynet_a (largest) ... tinynet_e (smallest, fastest)
MODES      = ['frozen', 'partial', 'full']
DATASETS   = ['IDIAP', 'PLUS', 'SCUT']
SIZES      = ['BA', 'S1', 'S2']    # BA = 4 training folds, S1 = 2, S2 = 1
FOLDS      = [1, 2, 3, 4, 5]
SEEDS      = [0]                   # add 1, 2 for mean over seeds (3x time)

EPOCHS     = 20                    # max epochs, early stopping on validation loss
PATIENCE   = 5
BATCH      = 32
LR_HEAD    = 1e-3
LR_BACKBONE = 1e-4                 # used by 'partial' and 'full'
WEIGHT_DECAY = 1e-4

NORM       = 'none'                # 'none' | 'zscore' (per-image brightness normalisation = brightness-cue control)
PLUS_MASK_LEFT = 0                 # e.g. 53 -> black out the printed PLUS label (left 53 px)
SAVE_PREDICTIONS = True

QUICK = False                      # True = smoke test
if QUICK:
    SIZES, FOLDS, SEEDS, EPOCHS, PATIENCE = ['BA'], [1], [0], 3, 3

RUNS_CSV  = f'{OUT}/tinynet_runs.csv'
SUM_CSV   = f'{OUT}/tinynet_summary.csv'
PRED_CSV  = f'{OUT}/tinynet_predictions.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
print('device', DEVICE)

## 2. Data, splits, metrics
The splits use the same code and seeds as `pad.py`, so the results compare directly with Experiment 1. Folds are subject-disjoint. The inner validation set (20 % of the **training** subjects) is used for early stopping and for the choice of k. The test fold is used only once, at the end.

In [ ]:
SIZE_FOLDS = {'BA': 4, 'S1': 2, 'S2': 1}
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
K_GRID = [1, 3, 5, 7, 9, 15]

def make_split(meta_ds, test_fold, size, seed, val_frac=0.2):
    rng = np.random.default_rng(1000 * seed + test_fold)
    others = [f for f in range(1, 6) if f != test_fold]
    train_folds = sorted(rng.choice(others, SIZE_FOLDS[size], replace=False).tolist())
    pool = meta_ds[meta_ds.fold.isin(train_folds)]
    subjects = np.array(sorted(pool.subject.unique()))
    n_val = max(1, int(round(val_frac * len(subjects))))
    val_subj = set(rng.choice(subjects, n_val, replace=False).tolist())
    fit = pool.index[~pool.subject.isin(val_subj)].to_numpy()
    val = pool.index[pool.subject.isin(val_subj)].to_numpy()
    test = meta_ds.index[meta_ds.fold == test_fold].to_numpy()
    assert not set(meta_ds.loc[test, 'subject']) & set(pool.subject), 'subject leak train/test'
    assert not set(meta_ds.loc[fit, 'subject']) & val_subj, 'subject leak fit/val'
    return fit, val, test, train_folds

def load_images(meta_ds, mask_left=0, norm='none'):
    arr = np.stack([np.asarray(Image.open(os.path.join(DATA, p)).convert('L')) for p in meta_ds.path])
    if mask_left > 0:
        arr[:, :, :mask_left] = 0
    if norm == 'zscore':
        f = arr.astype(np.float32)
        m = f.mean(axis=(1, 2), keepdims=True); s = f.std(axis=(1, 2), keepdims=True)
        arr = np.clip(128 + 48 * (f - m) / (s + 1e-6), 0, 255).astype(np.uint8)
    return arr

def pad_metrics(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    att, bf = y == 1, y == 0
    apcer = ((pred == 0) & att).sum() / max(att.sum(), 1)
    bpcer = ((pred == 1) & bf).sum() / max(bf.sum(), 1)
    return dict(APCER=100 * apcer, BPCER=100 * bpcer, ACER=100 * (apcer + bpcer) / 2)

def score_metrics(y, s):
    from sklearn.metrics import roc_auc_score, roc_curve
    y, s = np.asarray(y), np.asarray(s, float)
    fpr, tpr, _ = roc_curve(y, s); apcer = 1 - tpr
    i = int(np.argmin(np.abs(fpr - apcer)))
    ok = apcer <= 0.05
    a, b = s[y == 1], s[y == 0]
    return dict(EER=100 * (fpr[i] + apcer[i]) / 2, AUC=100 * roc_auc_score(y, s),
                BPCER20=100 * (fpr[ok].min() if ok.any() else 1.0),
                dprime=abs(a.mean() - b.mean()) / np.sqrt(0.5 * (a.var() + b.var()) + 1e-12))

def l2n(x): return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def knn_eval(X, y, fit, val, test):
    # k chosen on fit->val by ACER, then refit on fit+val; score = d_BF / (d_BF + d_PAI)
    from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
    X = l2n(X); best = None
    for k in K_GRID:
        if k > len(fit): break
        acer = pad_metrics(y[val], KNeighborsClassifier(k).fit(X[fit], y[fit]).predict(X[val]))['ACER']
        if best is None or acer < best[1]: best = (k, acer)
    k = best[0]; pool = np.concatenate([fit, val])
    pred = KNeighborsClassifier(k).fit(X[pool], y[pool]).predict(X[test])
    d = [NearestNeighbors(n_neighbors=min(k, (y[pool] == c).sum())).fit(X[pool][y[pool] == c]).kneighbors(X[test])[0].mean(1)
         for c in (0, 1)]
    return k, pred, d[0] / (d[0] + d[1] + 1e-12)

meta = pd.read_csv(f'{DATA}/metadata.csv', dtype={'subject': str})
print(meta.groupby(['dataset', 'label']).size().unstack().rename(columns={0: 'bona fide', 1: 'PAI'}))

## 3. Model and training
- Grey image → replicated to 3 channels → ImageNet normalisation. Images keep their native size (height 160), and global average pooling handles any width.
- Head: dropout 0.2 → linear 256 → ReLU → linear 1 (BCE loss).
- AdamW with **two learning rates** (head 1e-3, backbone 1e-4), 1 warm-up epoch, then cosine decay, mixed precision.
- Augmentation on the GPU: brightness/contrast ±10 %, horizontal flip, rotation ±3°, shift ±4 %.
- Early stopping on the validation loss. The best epoch's weights are restored.

In [ ]:
class PADNet(nn.Module):
    def __init__(self, mode):
        super().__init__()
        self.backbone = timm.create_model(MODEL, pretrained=True, num_classes=0)   # -> pooled features
        self.dim = self.backbone.num_features
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.dim, 256), nn.ReLU(), nn.Linear(256, 1))
        self.mode = mode
        for p in self.backbone.parameters():
            p.requires_grad = mode == 'full'
        if mode == 'partial':                       # last 2 block stages + final 1x1 conv + bn
            for m in list(self.backbone.blocks)[-2:] + [self.backbone.conv_head, self.backbone.bn2]:
                for p in m.parameters(): p.requires_grad = True
        # modules whose BN statistics must stay fixed (fully frozen sub-modules)
        self.frozen_mods = [m for m in self.backbone.modules()
                            if isinstance(m, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in m.parameters())]

    def train_mode(self):
        self.train()
        for m in self.frozen_mods: m.eval()

    def embed(self, x): return self.backbone(x)
    def forward(self, x): return self.head(self.backbone(x)).squeeze(1)

def to_batch(u8, augment=False):
    x = torch.from_numpy(u8).to(DEVICE, non_blocking=True).float().div_(255).unsqueeze(1)
    if augment:
        B, _, H, W = x.shape
        b = 1 + (torch.rand(B, 1, 1, 1, device=DEVICE) - 0.5) * 0.2
        c = 1 + (torch.rand(B, 1, 1, 1, device=DEVICE) - 0.5) * 0.2
        m = x.mean(dim=(2, 3), keepdim=True)
        x = ((x - m) * c + m) * b
        flip = torch.rand(B, device=DEVICE) < 0.5
        x[flip] = x[flip].flip(-1)
        ang = (torch.rand(B, device=DEVICE) - 0.5) * 2 * np.deg2rad(3)
        tx, ty = [(torch.rand(B, device=DEVICE) - 0.5) * 0.16 for _ in range(2)]
        cos, sin = torch.cos(ang), torch.sin(ang)
        theta = torch.stack([torch.stack([cos, -sin * H / W, tx], 1), torch.stack([sin * W / H, cos, ty], 1)], 1)
        x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), padding_mode='border', align_corners=False).clamp_(0, 1)
    x = x.expand(-1, 3, -1, -1)
    return (x - torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1)) / torch.tensor(STD, device=DEVICE).view(1, 3, 1, 1)

@torch.no_grad()
def infer(model, imgs, bs=64):
    model.eval(); emb, prob = [], []
    with torch.autocast(DEVICE.type, enabled=AMP):
        for i in range(0, len(imgs), bs):
            e = model.embed(to_batch(imgs[i:i + bs]))
            emb.append(e.float().cpu().numpy())
            prob.append(torch.sigmoid(model.head(e).squeeze(1).float()).cpu().numpy())
    return np.concatenate(emb), np.concatenate(prob)

def backbone_vector(model):
    return torch.cat([p.detach().float().flatten().cpu() for p in model.backbone.parameters()])

def train(model, imgs, y, fit, val, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model.to(DEVICE)
    body = [p for p in model.backbone.parameters() if p.requires_grad]
    groups = [{'params': model.head.parameters(), 'lr': LR_HEAD}] + ([{'params': body, 'lr': LR_BACKBONE}] if body else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    spe = int(np.ceil(len(fit) / BATCH)); total = EPOCHS * spe
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / spe if s < spe else 0.5 * (1 + np.cos(np.pi * (s - spe) / max(1, total - spe))))
    scaler = torch.amp.GradScaler(enabled=AMP)
    yt = torch.tensor(y, dtype=torch.float32)
    best, best_state, bad, best_ep, hist = 1e9, None, 0, 0, []
    for ep in range(EPOCHS):
        model.train_mode(); order = rng.permutation(fit); tl = []
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            if len(idx) < 2: continue
            with torch.autocast(DEVICE.type, enabled=AMP):
                loss = F.binary_cross_entropy_with_logits(model(to_batch(imgs[idx], True)).float(), yt[idx].to(DEVICE))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tl.append(loss.item())
        _, p = infer(model, imgs[val]); p = np.clip(p, 1e-6, 1 - 1e-6)
        vl = float(-np.mean(y[val] * np.log(p) + (1 - y[val]) * np.log(1 - p)))
        hist.append((round(float(np.mean(tl)), 4), round(vl, 4)))
        if vl < best - 1e-4:
            best, bad, best_ep = vl, 0, ep + 1
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return best_ep, best, hist

for mode in MODES:   # sanity check: what is trainable in each mode
    m = PADNet(mode)
    nb = sum(p.numel() for p in m.backbone.parameters()); nt = sum(p.numel() for p in m.backbone.parameters() if p.requires_grad)
    print(f'{mode:8s} backbone params {nb:,} | trainable in backbone {nt:,} ({100 * nt / nb:.0f} %) | feature dim {m.dim}')
del m

## 4. Run the experiment
Each finished run is appended to `tinynet_runs.csv` right away. If the session stops, run this cell again: finished runs are skipped.
Every line shows the k-NN result on the fine-tuned embeddings and the result of the trained classifier head.

In [ ]:
done = set(pd.read_csv(RUNS_CSV).key) if os.path.exists(RUNS_CSV) else set()
variant = f'{NORM}' + (f'-mask{PLUS_MASK_LEFT}' if PLUS_MASK_LEFT else '')
t_all = time.time()
for ds in DATASETS:
    mds = meta[meta.dataset == ds].reset_index(drop=True)
    imgs = load_images(mds, PLUS_MASK_LEFT if ds == 'PLUS' else 0, NORM)
    y = mds.label.to_numpy()
    print(f'\n===== {ds}: {imgs.shape} =====')
    for size in SIZES:
        for seed in SEEDS:
            for fold in FOLDS:
                fit, val, test, tf = make_split(mds, fold, size, seed)
                for mode in MODES:
                    key = f'{MODEL}|{mode}|{ds}|{size}|f{fold}|s{seed}|{variant}'
                    if key in done: continue
                    t = time.time()
                    model = PADNet(mode)
                    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
                    w0 = backbone_vector(model)
                    ep, vloss, hist = train(model, imgs, y, fit, val, seed)
                    change = float((backbone_vector(model) - w0).norm() / w0.norm())
                    X, prob = infer(model, imgs)
                    k, kpred, kscore = knn_eval(X, y, fit, val, test)
                    mk, sk = pad_metrics(y[test], kpred), score_metrics(y[test], kscore)
                    mh, sh = pad_metrics(y[test], (prob[test] > 0.5).astype(int)), score_metrics(y[test], prob[test])
                    rec = dict(key=key, model=MODEL, mode=mode, dataset=ds, size=size, fold=fold, seed=seed, variant=variant,
                               train_folds=str(tf), n_fit=len(fit), n_val=len(val), n_test=len(test),
                               trainable_params=n_train, backbone_rel_change=change, best_epoch=ep, epochs_run=len(hist),
                               val_loss=vloss, k=k,
                               **{f'knn_{a}': v for a, v in {**mk, **sk}.items()},
                               **{f'head_{a}': v for a, v in {**mh, **sh}.items()},
                               seconds=round(time.time() - t, 1), history=json.dumps(hist))
                    pd.DataFrame([rec]).to_csv(RUNS_CSV, mode='a', header=not os.path.exists(RUNS_CSV), index=False)
                    if SAVE_PREDICTIONS:
                        pd.DataFrame(dict(key=key, path=mds.path[test].values, label=y[test], head_prob=prob[test],
                                          knn_score=kscore, knn_pred=kpred)).to_csv(
                            PRED_CSV, mode='a', header=not os.path.exists(PRED_CSV), index=False)
                    print(f'{ds:5s} {size} f{fold} s{seed} {mode:7s} ep={ep:2d}/{len(hist):2d} Δw={change:.4f} | '
                          f'kNN ACER={mk["ACER"]:5.2f} EER={sk["EER"]:5.2f} | head ACER={mh["ACER"]:5.2f} EER={sh["EER"]:5.2f} '
                          f'd\'={sh["dprime"]:5.2f} [{time.time() - t:.0f}s]')
                    del model
                    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
print(f'\nall done in {(time.time() - t_all) / 60:.1f} min -> {RUNS_CSV}')

## 5. Summary CSV (mean ± std over folds and seeds)

In [ ]:
r = pd.read_csv(RUNS_CSV)
cols = ['knn_ACER', 'knn_EER', 'knn_AUC', 'knn_dprime', 'head_APCER', 'head_BPCER', 'head_ACER', 'head_EER',
        'head_AUC', 'head_BPCER20', 'head_dprime', 'backbone_rel_change', 'best_epoch', 'seconds']
g = r.groupby(['model', 'variant', 'dataset', 'mode', 'size'])
s = g[cols].agg(['mean', 'std']).round(4)
s.columns = [f'{c}_{st}' for c, st in s.columns]
s['runs'] = g.size()
s = s.reset_index()
s['size'] = pd.Categorical(s['size'], ['BA', 'S1', 'S2'], ordered=True)
s['mode'] = pd.Categorical(s['mode'], ['frozen', 'partial', 'full'], ordered=True)
s = s.sort_values(['variant', 'dataset', 'size', 'mode'])
s.to_csv(SUM_CSV, index=False)
print('written:', SUM_CSV, '|', len(r), 'runs')
for metric in ['head_ACER_mean', 'head_EER_mean', 'knn_ACER_mean', 'head_dprime_mean', 'backbone_rel_change_mean']:
    print(f'\n===== {metric} =====')
    print(s.pivot_table(index=['dataset', 'mode'], columns='size', values=metric, observed=True).round(3).to_string())

## 6. Proof that the backbone was really fine-tuned
`backbone_rel_change` must be **0 for `frozen`** and clearly **> 0 for `partial` / `full`**. The training/validation loss history of one run per mode is shown below.

In [ ]:
print(r.groupby('mode')[['trainable_params', 'backbone_rel_change', 'best_epoch']].agg(['min', 'mean', 'max']).round(4).to_string())
for mode, row in r.groupby('mode').head(1).set_index('mode').iterrows():
    print(f'\n{mode} ({row.key}) epoch: (train loss, val loss)')
    for i, h in enumerate(json.loads(row.history), 1): print(f'  {i:2d}: {h}')

## 7. Save
Click *Save Version → Save & Run All (Commit)*. When it finishes, open the version, go to the *Output* tab, and download `tinynet_runs.csv`, `tinynet_summary.csv` and `tinynet_predictions.csv` into `D:\AIS\IPI\results`.